In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

#1.




In [16]:

def load_data(path, rent_is_manen=True, treat_pet_consult_as_allowed=False):
    df = pd.read_csv(path)

    rename_map = {
        "所在地": "location",
        "管理費_共益費": "management_fee",
        "向き": "direction",
        "階": "floor_raw",
        "条件": "conditions",
    }
    df = df.rename(columns=rename_map).copy()

    # Floor
    if "floor_raw" in df.columns:
        df["floor"] = (
            df["floor_raw"].astype(str).str.extract(r"(\d+)", expand=False)
        )
        df["floor"] = pd.to_numeric(df["floor"], errors="coerce").fillna(0)
    else:
        df["floor"] = 0

    # Pet allowed
    if "conditions" in df.columns:
        cond = df["conditions"].astype(str)
        if treat_pet_consult_as_allowed:
            df["pet_allowed"] = cond.apply(
                lambda x: 1 if ("ペット" in x and "不可" not in x) else 0
            )
        else:
            df["pet_allowed"] = cond.apply(
                lambda x: 1 if ("ペット" in x and "不可" not in x and "可" in x) else 0
            )
    else:
        df["pet_allowed"] = 0

    numeric_cols = [
        "rent", "management_fee", "area_m2",
        "building_age", "walking_distance_to_station"
    ]

    for col in numeric_cols:
        if col not in df.columns:
            df[col] = 0
            continue

        s = (
            df[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("¥", "", regex=False)
            .str.extract(r"([-]?\d+\.?\d*)", expand=False)
        )
        df[col] = pd.to_numeric(s, errors="coerce")

    # Fill only numeric NaN
    df[numeric_cols + ["floor", "pet_allowed"]] = df[numeric_cols + ["floor", "pet_allowed"]].fillna(0)

    # Normalize units for total cost
    rent_yen = df["rent"] * 10000 if rent_is_manen else df["rent"]
    total_cost = (rent_yen + df["management_fee"]).replace(0, np.nan)

    df["cost_performance"] = (df["area_m2"] / total_cost).fillna(0)

    return df

#2. CREATE HEURISTIC MATCH SCORE LABEL (0–100)

In [9]:
def compute_heuristic_match_score(df):
    df = df.copy()

    # Normalize
    def norm(col):
        mn, mx = df[col].min(), df[col].max()
        if mx - mn == 0:
            return df[col] * 0
        return (df[col] - mn) / (mx - mn)

    df["norm_rent"] = norm("rent")
    df["norm_distance"] = norm("walking_distance_to_station")
    df["norm_cp"] = norm("cost_performance")

    df["budget_score"] = 1 - df["norm_rent"]
    df["distance_score"] = 1 - df["norm_distance"]

    ideal_area = df["area_m2"].median()
    df["size_score"] = np.exp(-abs(df["area_m2"] - ideal_area) / ideal_area)

    df["age_score"] = 1 - (df["building_age"] / (df["building_age"].max() + 1))

    df["match_score"] = (
        df["budget_score"] * 0.30 +
        df["distance_score"] * 0.25 +
        df["size_score"] * 0.20 +
        df["age_score"] * 0.10 +
        df["norm_cp"] * 0.15
    ) * 100

    return df

#3. SELECT FEATURES FOR TRAINNING

In [10]:
def extract_features(df):
    feature_cols = [
        "rent",
        "management_fee",
        "area_m2",
        "building_age",
        "walking_distance_to_station",
        "floor",
        "pet_allowed",
        "cost_performance"
    ]

    X = df[feature_cols]
    y = df["match_score"]

    return X, y, feature_cols

#4. TRAIN MODEL

In [11]:
def train_model(X, y):
    model = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )
    model.fit(X, y)
    return model


In [ ]:
def save_model(model, feature_cols, path="yodogawa_match_model.pkl"):
    obj = {
        "model": model,
        "feature_cols": feature_cols     
    }
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print("MODEL + FEATURE_COLS saved!")

In [18]:
if __name__ == "__main__":
    print("Loading data...")
    df = load_data(r".\data\yodogawa_feature_eng.csv")

    print("Generating match score labels...")
    df = compute_heuristic_match_score(df)

    print("Extracting features...")
    X, y, feature_cols = extract_features(df)

    print("Training model...")
    model = train_model(X, y)

    print("Saving model...")
    save_model(
        model,
        feature_cols,
        r".\model\yodogawa_match_model.pkl"
    )

    print("\n🎉 Training completed successfully!")
    print("👉 Saved as: yodogawa_match_model.pkl")

Loading data...
Generating match score labels...
Extracting features...
Training model...
Saving model...
MODEL + FEATURE_COLS saved!

🎉 Training completed successfully!
👉 Saved as: yodogawa_match_model.pkl
